In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
# OBTENEMOS LOS DATOS DE LA CAPA SILVER
gold_silver = spark.table("spotify_catalog.silver.spotify_tracks")

display(gold_silver.limit(10))

In [0]:
dim_album_base  = (
    gold_silver
    .select(
        "album_id",
        "album_name",
        "album_type",
        "release_date",
        "release_date_precision",
        "release_date_clean",
        "album_total_tracks",
        "artist_id"
    )
    .filter(col("album_id").isNotNull())
    .dropDuplicates(["album_id"])
)


In [0]:
window_album = Window.orderBy("album_id")

dim_album = (
    dim_album_base
    .withColumn(
        "sk_album",
        row_number().over(window_album)
    )
    .select(
        "sk_album",
        "album_id",
        "album_name",
        "album_type",
        "release_date",
        "release_date_precision",
        "release_date_clean",
        "album_total_tracks"
    )
)


In [0]:
(
    dim_album
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "spotify_catalog.gold.dim_album"
    )
)